# 🗣️ Módulo 12 — Orchestration: Group Chat

> **Objetivo:** Coordinar una conversación grupal entre varios agentes con un orquestador que decide quién habla a continuación.

📚 **Doc oficial:** [Group Chat Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/group-chat?pivots=programming-language-python)

## ¿Qué es Group Chat Orchestration?

Topología en **estrella** — todos los agentes hablan a través de un orquestador central que
decide qué agente toma el turno siguiente, basándose en una función de selección o en otro agente.

```
              ┌───────────┐
              │Orchestrator│
              └─┬───┬───┬─┘
        ┌───────┘   │   └────────┐
   ┌────▼───┐   ┌───▼───┐   ┌───▼────┐
   │ Agent A│   │Agent B│   │ Agent C│
   └────────┘   └───────┘   └────────┘
```

Ideal para:
- **Refinamiento iterativo** — round-robin de revisores
- **Colaboración multi-perspectiva** — Researcher + Writer + Critic
- Debates / brainstorming estructurado

## Selectores disponibles

| Selector | API | Cuándo usar |
|----------|-----|-------------|
| Round-robin | `selection_func=round_robin_selector` | Todos hablan por turno |
| Custom Python | `selection_func=lambda state: ...` | Lógica determinista basada en estado |
| Agent-based | `orchestrator_agent=Agent(...)` | LLM decide quién habla (con contexto) |


## ⚙️ Setup

In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


## 1️⃣ Round-robin selector

El selector más simple — recorre los participantes en orden. Útil cuando todos deben
contribuir por igual.


In [ ]:
from agent_framework.orchestrations import GroupChatBuilder, GroupChatState

client = create_chat_client()

researcher = Agent(
    client=client,
    name="Researcher",
    description="Collects relevant background information.",
    instructions=(
        "Gather concise facts that help answer the question. Be brief and factual."
    ),
)

writer = Agent(
    client=client,
    name="Writer",
    description="Synthesizes polished answers using gathered information.",
    instructions=(
        "Compose clear, structured answers using any notes provided. Be comprehensive."
    ),
)


def round_robin_selector(state: GroupChatState) -> str:
    """Selecciona al siguiente speaker en round-robin."""
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]


workflow = GroupChatBuilder(
    participants=[researcher, writer],
    termination_condition=lambda conversation: len(conversation) >= 4,
    selection_func=round_robin_selector,
).build()

print("✅ Group chat configurado con round-robin (máx 4 mensajes)")


## 2️⃣ Ejecutar y observar la conversación

Activa `stream=True` para ver cada actualización a medida que llega.
El output terminal es un `AgentResponse` con el resumen del orquestador.


In [ ]:
from agent_framework import AgentResponse, AgentResponseUpdate

task = "What are the key benefits of async/await in Python?"

print(f"Task: {task}\n{'=' * 80}\n")

last_author = None
final_response: AgentResponse | None = None

async for event in workflow.run(task, stream=True):
    if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
        author = event.data.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"[{author}]:", end=" ", flush=True)
            last_author = author
        print(event.data.text, end="", flush=True)
    elif event.type == "output" and isinstance(event.data, AgentResponse):
        final_response = event.data

print(f"\n\n{'=' * 80}\n✅ Conversación completada.")


## 3️⃣ Selector custom basado en contenido

`GroupChatState.conversation` te da acceso a todos los mensajes hasta ahora.
Puedes implementar lógica inteligente — por ejemplo, mantener al `Researcher` hablando
hasta que diga "I have finished", luego pasar al `Writer`.


In [ ]:
def smart_selector(state: GroupChatState) -> str:
    """Selecciona basado en el contenido del último mensaje."""
    conversation = state.conversation
    last = conversation[-1] if conversation else None

    if not last:
        return "Researcher"

    last_text = (last.text or "").lower()
    if "finished" in last_text and last.author_name == "Researcher":
        return "Writer"

    return "Researcher"


workflow = GroupChatBuilder(
    participants=[researcher, writer],
    termination_condition=lambda conv: len(conv) >= 6,
    selection_func=smart_selector,
).build()

last_author = None
async for event in workflow.run(
    "Explain monad transformers and end with 'I have finished' when you have enough info.",
    stream=True,
):
    if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
        author = event.data.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"[{author}]:", end=" ", flush=True)
            last_author = author
        print(event.data.text, end="", flush=True)
print()


## 4️⃣ Orquestador basado en agente (LLM)

Para conversaciones más complejas, deja que **otro agente** decida quién habla a continuación.
El orquestador es un `Agent` completo con sus propias instructions y tools.


In [ ]:
orchestrator_agent = Agent(
    client=client,
    name="Orchestrator",
    description="Coordinates multi-agent collaboration by selecting speakers",
    instructions=(
        "You coordinate a team conversation to solve the user's task.\n\n"
        "Guidelines:\n"
        "- Start with Researcher to gather information\n"
        "- Then have Writer synthesize the final answer\n"
        "- Only finish after both have contributed meaningfully"
    ),
)

workflow = GroupChatBuilder(
    participants=[researcher, writer],
    # Termina cuando hay al menos 4 mensajes assistant
    termination_condition=lambda msgs: sum(1 for m in msgs if m.role == "assistant") >= 4,
    orchestrator_agent=orchestrator_agent,
).build()

last_author = None
async for event in workflow.run(
    "What are the trade-offs of using SQLite vs PostgreSQL for a SaaS startup?",
    stream=True,
):
    if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
        author = event.data.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"[{author}]:", end=" ", flush=True)
            last_author = author
        print(event.data.text, end="", flush=True)
print("\n\n✅ Conversación con orquestador agente completada.")


## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Round-robin | `selection_func=round_robin_selector` (con `GroupChatState`) |
| Selector custom | `selection_func=lambda state: "AgentName"` |
| Orquestador LLM | `orchestrator_agent=Agent(...)` |
| Condición de terminación | `termination_condition=lambda conv: len(conv) >= N` |
| Outputs intermedios | `intermediate_output_from=[a, b]` |
| Streaming | `async for event in workflow.run(task, stream=True):` |

## 📊 Comparación de patrones de orquestación

| Patrón | Cuándo usar |
|--------|-------------|
| **Sequential** (mod 09) | Pipeline lineal con etapas claras |
| **Concurrent** (mod 10) | Múltiples perspectivas en paralelo |
| **Handoff** (mod 11) | Delegación dinámica entre especialistas |
| **Group Chat** (mod 12) | Conversación colaborativa con turnos |

**Siguiente módulo →** [Módulo 13 — Workflows: Ejecutores (low-level)](./13_workflows_executors.ipynb)
